<a href="https://colab.research.google.com/github/mhmd2015/AI/blob/main/LLM_from_scratch_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Testing environment

In [1]:
"""
GPU & PyTorch verification script.

Run once after installation to confirm everything works.
Prints your full stack and tells you which advanced features
your hardware supports.
"""

import torch
import sys


# ---------- Compute capability reference ----------
COMPUTE_CAPABILITY_INFO = {
    # (major, minor): (architecture name, GPU family, feature notes)
    (3, 5): ("Kepler", "GTX 700-series, Tesla K-series", "Very old; PyTorch dropped support after 2.0"),
    (3, 7): ("Kepler", "Tesla K80", "Very old; minimal modern support"),
    (5, 0): ("Maxwell", "GTX 750 / 750 Ti", "Old; supported but no modern features"),
    (5, 2): ("Maxwell", "GTX 9-series, Titan X", "Old; supported but slow for ML"),
    (6, 0): ("Pascal", "Tesla P100", "Datacenter; FP16 supported"),
    (6, 1): ("Pascal", "GTX 10-series, Titan Xp", "Common gaming card; FP16 in software"),
    (7, 0): ("Volta", "Tesla V100", "First Tensor Cores; major ML jump"),
    (7, 5): ("Turing", "GTX 16-series, RTX 20-series", "Tensor Cores; no BF16; what most hobbyists have"),
    (8, 0): ("Ampere", "A100", "BF16 native; Flash Attention supported"),
    (8, 6): ("Ampere", "RTX 30-series, A40", "BF16 native; Flash Attention supported"),
    (8, 9): ("Ada Lovelace", "RTX 40-series, L40", "Latest BF16/FP8; great for ML"),
    (9, 0): ("Hopper", "H100, H200", "FP8 training; cutting edge datacenter"),
    (10, 0): ("Blackwell", "RTX 50-series, B100", "Newest; FP4 inference, FP8 training"),
}


def lookup_capability(major, minor):
    """Return architecture info for a given compute capability, or a generic note."""
    key = (major, minor)
    if key in COMPUTE_CAPABILITY_INFO:
        return COMPUTE_CAPABILITY_INFO[key]
    return ("Unknown", "Unrecognized GPU", "Compute capability not in lookup table")


def feature_support(major, minor):
    """Tell the user what modern ML features their card supports."""
    cc = major + minor / 10  # e.g. 7.5
    features = []

    # FP16 (half precision)
    if cc >= 5.3:
        features.append(("FP16 mixed precision", True, "halves activation memory, faster matmul"))
    else:
        features.append(("FP16 mixed precision", False, "GPU too old"))

    # Tensor Cores (specialized matmul hardware)
    if cc >= 7.0:
        features.append(("Tensor Cores", True, "huge speedup on FP16 matmul"))
    else:
        features.append(("Tensor Cores", False, "matmul runs on regular CUDA cores"))

    # BF16 (bfloat16) — preferred for training stability
    if cc >= 8.0:
        features.append(("BF16 training", True, "stable training, no loss scaling needed"))
    else:
        features.append(("BF16 training", False, "use FP16 mixed precision instead"))

    # Flash Attention (memory-efficient attention)
    if cc >= 8.0:
        features.append(("Flash Attention", True, "essential for long-context training"))
    else:
        features.append(("Flash Attention", False, "use standard attention; fine for ≤512 tokens"))

    # FP8 (8-bit floats, very new)
    if cc >= 8.9:
        features.append(("FP8 training", True, "2× faster than FP16 if your model supports it"))
    else:
        features.append(("FP8 training", False, "Hopper or newer required"))

    return features


# ---------- Run the checks ----------
print("=" * 60)
print(" PyTorch & GPU Verification")
print("=" * 60)

print(f"\nPython version : {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    print("\n⚠  WARNING: CUDA not available. Training will run on CPU and be VERY slow.")
    print("   Likely causes:")
    print("     - PyTorch installed without CUDA support (CPU-only wheel)")
    print("     - NVIDIA driver too old or missing")
    print("     - GPU not detected (check `nvidia-smi`)")
    sys.exit(1)


# ---------- GPU details ----------
print(f"\nCUDA runtime  : {torch.version.cuda}")
print(f"  ↑ This is the CUDA version PyTorch was built against — bundled inside the wheel.")
print(f"    It is NOT the same as the 'CUDA Version' shown in `nvidia-smi`,")
print(f"    which is the *maximum* version your driver can support.")
print(f"    As long as nvidia-smi's number ≥ this number, you're fine.")

props = torch.cuda.get_device_properties(0)
print(f"\nGPU            : {torch.cuda.get_device_name(0)}")
print(f"Total VRAM     : {props.total_memory / 1e9:.2f} GB")
print(f"Compute capability: {props.major}.{props.minor}")


# ---------- Architecture explanation ----------
arch, family, notes = lookup_capability(props.major, props.minor)
print(f"  ↑ Architecture: {arch} ({family})")
print(f"    {notes}")
print()
print(f"  About 'compute capability':")
print(f"    Every NVIDIA GPU has a version number indicating its hardware generation.")
print(f"    Higher numbers = newer architecture = more ML features supported.")
print(f"    Anything ≥ 7.0 (Volta or newer) has Tensor Cores and trains modern models well.")
print(f"    Anything ≥ 8.0 (Ampere or newer) supports BF16 and Flash Attention.")


# ---------- Feature support summary ----------
print(f"\nFeatures supported on this GPU:")
print(f"  {'Feature':<25} {'Supported':<12} Notes")
print(f"  {'-' * 25:<25} {'-' * 12:<12} {'-' * 40}")
for feature, supported, note in feature_support(props.major, props.minor):
    mark = "✓ yes" if supported else "✗ no"
    print(f"  {feature:<25} {mark:<12} {note}")


# ---------- Real GPU computation ----------
print(f"\nRunning a real computation on the GPU to confirm it works...")
x = torch.randn(1000, 1000, device='cuda')
y = x @ x.T
torch.cuda.synchronize()  # wait for the GPU to finish
print(f"  Matrix multiply succeeded. Output shape: {y.shape}")

allocated_mb = torch.cuda.memory_allocated() / 1e6
total_mb = props.total_memory / 1e6
print(f"  Memory allocated: {allocated_mb:.1f} MB / {total_mb:.0f} MB total ({100*allocated_mb/total_mb:.1f}%)")


# ---------- Final verdict ----------
print(f"\n{'=' * 60}")
if props.total_memory < 4e9:
    print(" ⚠  Your GPU has less than 4GB VRAM. Plan for very small models.")
elif props.total_memory < 7e9:
    print(" ✓  Your GPU is suitable for educational LLM training (10–30M params).")
elif props.total_memory < 13e9:
    print(" ✓  Your GPU is suitable for serious training (50–150M params).")
else:
    print(" ✓  Your GPU is large enough for substantial models (300M+ params).")
print(f"{'=' * 60}\n")

 PyTorch & GPU Verification

Python version : 3.12.13
PyTorch version: 2.10.0+cu128
CUDA available : True

CUDA runtime  : 12.8
  ↑ This is the CUDA version PyTorch was built against — bundled inside the wheel.
    It is NOT the same as the 'CUDA Version' shown in `nvidia-smi`,
    which is the *maximum* version your driver can support.
    As long as nvidia-smi's number ≥ this number, you're fine.

GPU            : NVIDIA A100-SXM4-40GB
Total VRAM     : 42.41 GB
Compute capability: 8.0
  ↑ Architecture: Ampere (A100)
    BF16 native; Flash Attention supported

  About 'compute capability':
    Every NVIDIA GPU has a version number indicating its hardware generation.
    Higher numbers = newer architecture = more ML features supported.
    Anything ≥ 7.0 (Volta or newer) has Tensor Cores and trains modern models well.
    Anything ≥ 8.0 (Ampere or newer) supports BF16 and Flash Attention.

Features supported on this GPU:
  Feature                   Supported    Notes
  ----------------

# Expolre Data

In [2]:
from datasets import load_dataset

# This downloads TinyStories the first time (~2GB) and caches it locally
# Subsequent runs will be instant
print("Loading TinyStories...")
dataset = load_dataset("roneneldan/TinyStories")

# A dataset has "splits" — typically train and validation
print(f"\nDataset structure: {dataset}")

# Look at the first story
print(f"\n--- First story ---")
print(dataset['train'][0]['text'])

# Look at the 100th story to see variety
print(f"\n--- Story #100 ---")
print(dataset['train'][100]['text'])

# How many stories do we have?
print(f"\nTotal training stories: {len(dataset['train']):,}")
print(f"Total validation stories: {len(dataset['validation']):,}")

Loading TinyStories...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

data/train-00003-of-00004-d243063613e5a0(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]


Dataset structure: DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})

--- First story ---
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.

--- Story #100 ---
There was a little girl with dark hair. Her name was 

In [3]:
from datasets import load_dataset

# This downloads TinyStories the first time (~2GB) and caches it locally
# Subsequent runs will be instant
print("Loading Quran...")
dataset = load_dataset("arbml/quran_uthmani")


# A dataset has "splits" — typically train and validation
print(f"\nDataset structure: {dataset}")

# Look at the first story
print(f"\n--- First story ---")
print(dataset['train'][0]['sentence'])

# Look at the 100th story to see variety
print(f"\n--- Story #100 ---")
print(dataset['train'][100]['sentence'])

# How many stories do we have?
print(f"\nTotal training stories: {len(dataset['train']):,}")
#print(f"Total validation stories: {len(dataset['validation']):,}")

Loading Quran...

Dataset structure: DatasetDict({
    train: Dataset({
        features: ['sorah', 'ayah', 'sentence'],
        num_rows: 6235
    })
})

--- First story ---
ٱلْحَمْدُ لِلَّهِ رَبِّ ٱلْعَٰلَمِينَ

--- Story #100 ---
وَلَن يَتَمَنَّوْهُ أَبَدًۢا بِمَا قَدَّمَتْ أَيْدِيهِمْ وَٱللَّهُ عَلِيمٌۢ بِٱلظَّٰلِمِينَ

Total training stories: 6,235


# Train Tokenizer

In [8]:
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

# 1. Load the dataset (already cached from last time, so this is instant)
print("Loading dataset...")
#dataset = load_dataset("roneneldan/TinyStories")
#dataset = load_dataset("arbml/quran_uthmani")
dataset = load_dataset("M-AI-C/quran_en_translations")

# 2. Create an iterator that yields text in batches
# We don't load all 2M stories into memory at once — we stream them
def batch_iterator(batch_size=1000):
    for i in range(0, len(dataset['train']), batch_size):
        #yield dataset['train'][i : i + batch_size]['text']
        #yield dataset['train'][i : i + batch_size]['sentence']
        yield dataset['train'][i : i + batch_size]['en-ahmedali']

# 3. Initialize an empty BPE tokenizer
tokenizer = Tokenizer(BPE(unk_token="<|unk|>"))

# 4. Configure it to work at the byte level (handles any Unicode safely)
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)
tokenizer.decoder = ByteLevelDecoder()

# 5. Set up the trainer with our desired vocabulary size
trainer = BpeTrainer(
    vocab_size=4096,
    special_tokens=["<|endoftext|>", "<|unk|>"],
    initial_alphabet=ByteLevel.alphabet(),
    show_progress=True,
)

# 6. Train!
print("Training tokenizer on TinyStories (this takes a few minutes)...")
tokenizer.train_from_iterator(
    batch_iterator(),
    trainer=trainer,
    length=len(dataset['train']),
)

# 7. Save it to disk so we don't have to retrain
#tokenizer.save("tinystories_tokenizer.json")
#tokenizer.save("quran_tokenizer.json")
tokenizer.save("quran2_tokenizer.json")
print(f"\nDone! Vocabulary size: {tokenizer.get_vocab_size()}")
print("Saved to tinystories_tokenizer.json")

Loading dataset...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-05459c7713a431(…):   0%|          | 0.00/35.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6235 [00:00<?, ? examples/s]

Training tokenizer on TinyStories (this takes a few minutes)...

Done! Vocabulary size: 4096
Saved to tinystories_tokenizer.json


# Generate Test

In [12]:
import sys
import torch
from model import GPT, GPTConfig
from pathlib import Path
try:
    from tokenizers import Tokenizer
except ModuleNotFoundError as exc:
    raise SystemExit(
        "Missing dependency: tokenizers\n"
        "Install it in the active uv environment with:\n"
        "  uv pip install tokenizers"
    ) from exc

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

# Load our tokenizer from Episode 2
#tokenizer_path = Path("tinystories_tokenizer.json")
#tokenizer_path = Path("quran_tokenizer.json")
tokenizer_path = Path("quran2_tokenizer.json")
tokenizer = Tokenizer.from_file(str(tokenizer_path))

# Build the (randomly initialized) model
model = GPT(GPTConfig())
model.eval()

# Encode a prompt
#prompt = "Once upon a time"
#prompt = " بسم الله الرحمن"
prompt = "In the name "

ids = tokenizer.encode(prompt).ids
idx = torch.tensor([ids], dtype=torch.long)  # shape (1, T)

# Generate 100 tokens with no training whatsoever
with torch.no_grad():
    output = model.generate(idx, max_new_tokens=100, temperature=1.0)

# Decode and print
text = tokenizer.decode(output[0].tolist())
print(text)

In the name  increasesiteWe altern�ised feed certainly recite discusoodWhere fulfil distance lightningSatan wished�palm susp share raised sor await dow u progen exclusive honourovedustordailndise staff grieve talk TorahHE portumn sudden offspringree Surely mom�D ransarersSuchig tw tongue ledgerY rightlyful prays hypocritesardedess moun lawful iniquitous swe Had t haughty concerhabit Mosque dispen dweWhatsoeverjust progenalle Isredly forbidden discoursesufficient�sentll� exaltedIT per make Such doubled L possessed infidels bl repent inher
